# Data Preparation

This notebook prepares the enriched homestay dataset for the recommendation system by:

1. Inspecting and validating the enriched dataset.
2. Handling missing values and data inconsistencies.
3. Creating amenity-based text features from binary amenity indicators.
4. Creating location-based text features from geographical information.
5. Combining descriptions, amenities, and location information into a unified feature representation.
6. Preparing explainability-related features for recommendation interpretation.
7. Saving the processed dataset for recommendation model development.

The prepared dataset will be used for TF-IDF vectorization, similarity computation, and explainable homestay recommendation generation.

In [1]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================

import pandas as pd
import numpy as np

In [2]:
# ==========================================
# LOAD ENRICHED DATASET
# ==========================================

df = pd.read_csv(
    "../data/processed/homestays_enriched.csv"
)

print(f"Dataset Shape: {df.shape}")

Dataset Shape: (813, 40)


In [3]:
# ==========================================
# DATASET INFORMATION
# ==========================================

print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 813 entries, 0 to 812
Data columns (total 40 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   homestay_id             813 non-null    int64  
 1   homestay_name           813 non-null    str    
 2   owner_name              813 non-null    str    
 3   category                813 non-null    str    
 4   district                813 non-null    str    
 5   block                   813 non-null    str    
 6   village                 813 non-null    str    
 7   owner_email             797 non-null    str    
 8   owner_mobile            813 non-null    int64  
 9   google_name             813 non-null    str    
 10  google_address          813 non-null    str    
 11  latitude                813 non-null    float64
 12  longitude               813 non-null    float64
 13  rating                  813 non-null    float64
 14  review_count            813 non-null    int64  
 15  

In [4]:
# ==========================================
# REMOVE LEADING/TRAILING SPACES
# ==========================================

for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].astype("string").str.strip()

In [5]:
# ==========================================
# REMOVE NEWLINES / EXTRA WHITESPACES
# ==========================================

text_cols = [
    "homestay_name",
    "owner_name",
    "village",
    "google_address"
]

for col in text_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(r"[\r\n]+", " ", regex=True)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

In [6]:
# ==========================================
# CHECK MISSING VALUES
# ==========================================

missing_summary = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
)

print("\nMissing Values")
print(missing_summary[missing_summary > 0])


Missing Values
owner_email    16
dtype: int64


In [7]:
# ==========================================
# STANDARDIZE CAPITALIZATION
# ==========================================

title_cols = [
    "homestay_name",
    "owner_name",
    "category",
    "district",
    "block",
    "village"
]

print("Before standardization:")
print(df[title_cols].head(3))

for col in title_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.title()

print("\nAfter standardization:")
print(df[title_cols].head(3))

Before standardization:
         homestay_name             owner_name category   district  \
0      Revere Homestay      Mr. Riwaj Pradhan   Silver  KALIMPONG   
1  Mansarover Homestay  Miss Tina Mani Gurung     Gold  KALIMPONG   
2     BETHANY HOMESTAY         ANUPAMA TAMANG   Silver  KALIMPONG   

          block                  village  
0  Municipality      8th Mile, Kalimpong  
1  Municipality    Chandralok, Kalimpong  
2   Kalimpong 1  Dr.GRAHAMS HOME BLOCK B  

After standardization:
         homestay_name             owner_name category   district  \
0      Revere Homestay      Mr. Riwaj Pradhan   Silver  Kalimpong   
1  Mansarover Homestay  Miss Tina Mani Gurung     Gold  Kalimpong   
2     Bethany Homestay         Anupama Tamang   Silver  Kalimpong   

          block                  village  
0  Municipality      8Th Mile, Kalimpong  
1  Municipality    Chandralok, Kalimpong  
2   Kalimpong 1  Dr.Grahams Home Block B  


In [8]:
# ==========================================
# STANDARDIZE BLOCK NAMES
# ==========================================

print("Unique block values before:", sorted(df["block"].unique()))

block_mapping = {
    "Kalimpong 1": "Kalimpong I",
    "Kalimpong-1": "Kalimpong I",
    "Kalimpong-I": "Kalimpong I",
}

df["block"] = (
    df["block"]
    .replace(block_mapping)
)

print("Unique block values after:", sorted(df["block"].unique()))

Unique block values before: ['Gorubathan', 'Kalimpong 1', 'Kalimpong-1', 'Kalimpong-I', 'Lava', 'Municipality', 'Pedong']
Unique block values after: ['Gorubathan', 'Kalimpong I', 'Lava', 'Municipality', 'Pedong']


In [9]:
# ==========================================
# CREATE DUPLICATE CHECK KEY
# ==========================================

df["duplicate_check_key"] = (
    df["homestay_name"].fillna("") + "_" +
    df["village"].fillna("")
)

possible_duplicates = (
    df["duplicate_check_key"]
    .duplicated(keep=False)
    .sum()
)

print("Possible duplicate records:",
      possible_duplicates)

Possible duplicate records: 4


In [10]:
# ==========================================
# REVIEW AND RESOLVE DUPLICATE RECORDS
# ==========================================

flagged = df[df["duplicate_check_key"].duplicated(keep=False)].sort_values("duplicate_check_key")
print(f"Flagged near-duplicate records ({len(flagged)}):")
print(flagged[["homestay_id", "homestay_name", "village", "block"]].to_string())

before = len(df)
df = df.drop_duplicates(subset=["duplicate_check_key"], keep="first")

after = len(df)
print(f"\nRemoved {before - after} near-duplicate records, kept first occurrence by homestay_id.")

Flagged near-duplicate records (4):
     homestay_id        homestay_name       village   block
167          223  Mayellyang Homestay  Upper Pedong  Pedong
278          372  Mayellyang Homestay  Upper Pedong  Pedong
197          259       Sonam Homestay    Pithamchin  Pedong
693          996       Sonam Homestay    Pithamchin  Pedong

Removed 2 near-duplicate records, kept first occurrence by homestay_id.


In [11]:
# ==========================================
# VALIDATE COORDINATES
# ==========================================

if {"latitude", "longitude"}.issubset(df.columns):

    invalid_lat = (
        (df["latitude"] < -90) |
        (df["latitude"] > 90)
    )

    invalid_lon = (
        (df["longitude"] < -180) |
        (df["longitude"] > 180)
    )

    print(
        "Invalid coordinates:",
        (invalid_lat | invalid_lon).sum()
    )

Invalid coordinates: 0


In [12]:
# ==========================================
# VERIFY CATEGORY IS ALREADY CLEAN
# ==========================================
# Imputation now happens upstream in notebook 02, before price/amenity
# generation depend on it. This is a verification step, not a fix.

missing_category = df["category"].isna().sum()
assert missing_category == 0, f"Expected category to be fully imputed upstream, found {missing_category} missing."
print("Category verified: 0 missing (already imputed and standardized in notebook 02).")

Category verified: 0 missing (already imputed and standardized in notebook 02).


In [13]:
# ==========================================
# HANDLE DISTANCE FEATURES
# ==========================================

distance_columns = [
    "distance_to_deolo",
    "distance_to_durpin",
    "distance_to_town",
    "distance_to_lava",
    "distance_to_pedong",
    "distance_to_gorubathan",
    "distance_to_rishop",
    "distance_to_lolegaon"
]

for col in distance_columns:

    df[col] = df[col].fillna(
        df[col].median()
    )

In [14]:
# ==========================================
# HANDLE PROXIMITY FEATURES
# ==========================================

df["deolo_proximity"] = (
    df["deolo_proximity"]
    .fillna("Far")
)

df["durpin_proximity"] = (
    df["durpin_proximity"]
    .fillna("Far")
)

df["town_proximity"] = (
    df["town_proximity"]
    .fillna("Far")
)

In [15]:
# ==========================================
# CREATE PRICE BANDS
# ==========================================

def get_price_band(price):

    if price < 2000:
        return "budget"

    elif price < 3500:
        return "mid_range"

    else:
        return "premium"

df["price_band"] = df["price"].apply(get_price_band)

In [16]:
# ==========================================
# CREATE AMENITY TEXT FEATURE
# ==========================================

def create_amenity_text(row):

    amenities = []

    if row["wifi"] == 1:
        amenities.append("wifi")

    if row["parking"] == 1:
        amenities.append("parking")

    if row["breakfast"] == 1:
        amenities.append("breakfast")

    if row["mountain_view"] == 1:
        amenities.append("mountain_view")

    if row["room_service"] == 1:
        amenities.append("room_service")

    if row["bonfire_barbeque"] == 1:
        amenities.append("bonfire_barbeque")

    if row["pickup_dropoff_service"] == 1:
        amenities.append("pickup_dropoff_service")

    return " ".join(amenities)


df["amenity_text"] = df.apply(
    create_amenity_text,
    axis=1
)

df[[
    "homestay_name",
    "amenity_text"
]].head()

,homestay_name,amenity_text
0,Revere Homestay,wifi parking breakfast mountain_view
1,Mansarover Homestay,wifi parking breakfast room_service bonfire_ba...
2,Bethany Homestay,breakfast mountain_view
3,Bajarangi Homestay,parking breakfast mountain_view room_service
4,Relly View Homestay,wifi breakfast bonfire_barbeque pickup_dropoff...


In [17]:
# ==========================================
# CREATE LOCATION TEXT FEATURE
# ==========================================
# district is excluded -- it's "Kalimpong" for every single record in
# this single-district case study, so it carries zero discriminative
# signal for TF-IDF despite technically appearing as a "top matching
# term" in every explanation. block and village vary and carry real
# location signal, so those are kept.

df["location_text"] = (

    df["block"].astype(str)

    + " "

    + df["village"].astype(str)

)

In [18]:
# ==========================================
# CREATE PROXIMITY TEXT FEATURE
# ==========================================
# Compound location+proximity tokens (e.g. "durpin_very_close") keep the
# location and its proximity paired together -- unlike a bare "close",
# which doesn't say close to WHAT and had to be filtered out in
# notebook 05 as a near-universal, uninformative term.

proximity_columns = [
    "deolo_proximity", "durpin_proximity", "town_proximity",
    "lava_proximity", "pedong_proximity", "gorubathan_proximity",
    "rishop_proximity", "lolegaon_proximity",
]

def make_proximity_token(location_col, value):
    location_name = location_col.replace("_proximity", "")
    value_slug = str(value).lower().replace(" ", "_")
    return f"{location_name}_{value_slug}"

df["proximity_text"] = df[proximity_columns].apply(
    lambda row: " ".join(
        make_proximity_token(col, row[col]) for col in proximity_columns
    ),
    axis=1
)

In [19]:
# ==========================================
# CREATE FEATURE TEXT
# ==========================================

df["feature_text"] = (

    df["description"].astype(str)

    + " "

    + df["amenity_text"].astype(str)

    + " "

    + df["location_text"].astype(str)

    + " "

    + df["proximity_text"].astype(str)

    + " "

    + df["category"].astype(str)

    + " "

    + df["price_band"].astype(str)

)

**Resolved**: `feature_text` now includes `proximity_text` (compound
location+proximity tokens, e.g. "durpin_very_close"), alongside
`description + amenity_text + location_text + category + price_band`.
`rating` and `review_count` remain deliberately excluded -- review_count
in particular reflects popularity/exposure rather than the homestay's
actual content, and including it would shift this from a purely
content-based system toward a content+popularity hybrid. Worth one
explicit sentence in the Methodology chapter either way: proximity is
now genuinely part of the content profile; rating/review_count are
excluded by design, not by oversight.

In [20]:
# ==========================================
# FEATURE TEXT SAMPLE
# ==========================================

print(
    df.loc[0, "feature_text"]
)

Revere Homestay, located in 8th Mile, Kalimpong, block Municipality, is situated very close to town, moderately close to Deolo, and very close to Durpin. This Silver-rated homestay offers Wifi, Parking, Breakfast, and Mountain View. Rated 5.0 with 1 review, it provides a convenient stay with proximity to all three locations. wifi parking breakfast mountain_view Municipality 8Th Mile, Kalimpong deolo_moderately_close durpin_very_close town_very_close lava_far pedong_far gorubathan_far rishop_far lolegaon_far Silver mid_range


In [21]:
# ==========================================
# EXPLAINABILITY FEATURE SELECTION
# ==========================================

explainability_features = [

    "price",

    "rating",

    "review_count",

    "distance_to_deolo",

    "distance_to_durpin",

    "distance_to_town",

    "distance_to_lava",

    "distance_to_pedong",

    "distance_to_gorubathan",

    "distance_to_rishop",

    "distance_to_lolegaon"

]

print(
    df[explainability_features]
    .head()
)

   price  rating  review_count  distance_to_deolo  distance_to_durpin  \
0   2126     5.0             1               5.17                3.03   
1   3959     4.2            77               5.79                2.01   
2   1860     4.6            69               0.96                7.57   
3   2130     4.2             5               9.03                6.63   
4   2095     4.8           238               1.08                7.52   

   distance_to_town  distance_to_lava  distance_to_pedong  \
0              0.96             18.01               18.75   
1              1.08             18.60               19.30   
2              4.71             12.85               13.54   
3              7.13             19.19               19.62   
4              4.69             12.90               13.58   

   distance_to_gorubathan  distance_to_rishop  distance_to_lolegaon  
0                   26.25               19.75                 11.49  
1                   25.64               19.96         

## Model-Readiness Checks

In [22]:
# ==========================================
# MODEL-READINESS CHECKS
# ==========================================

# homestay_id must be a safe, unique lookup key for the recommendation
# system -- not homestay_name, which isn't guaranteed unique and is
# free-text entered by different people across the source registry
assert df["homestay_id"].is_unique, "homestay_id is not unique -- recommendation lookups would be ambiguous."
print(f"homestay_id verified unique across {len(df)} records.")

# feature_text is what TF-IDF vectorizes -- a null or empty value here
# means that row contributes nothing to the similarity space
empty_feature_text = (
    df["feature_text"].isna() | (df["feature_text"].astype(str).str.strip() == "")
).sum()
assert empty_feature_text == 0, f"{empty_feature_text} records have empty feature_text."
print(f"feature_text verified non-empty for all {len(df)} records.")

homestay_id verified unique across 811 records.
feature_text verified non-empty for all 811 records.


In [23]:
# ==========================================
# FINAL CLEANUP BEFORE SAVE
# ==========================================

# duplicate_check_key was only needed for the duplicate check above
df = df.drop(columns=["duplicate_check_key"])

print("\nMissing Values:")
print(
    df.isnull()
      .sum()
      .sort_values(ascending=False)
)



Missing Values:
owner_email               16
homestay_name              0
owner_name                 0
category                   0
homestay_id                0
district                   0
block                      0
village                    0
owner_mobile               0
google_name                0
google_address             0
latitude                   0
longitude                  0
rating                     0
review_count               0
distance_to_deolo          0
distance_to_durpin         0
distance_to_town           0
distance_to_lava           0
distance_to_pedong         0
distance_to_gorubathan     0
distance_to_rishop         0
distance_to_lolegaon       0
deolo_proximity            0
durpin_proximity           0
town_proximity             0
lava_proximity             0
pedong_proximity           0
gorubathan_proximity       0
rishop_proximity           0
lolegaon_proximity         0
price                      0
wifi                       0
parking                   

In [24]:
# ==========================================
# SAVE PREPARED DATASET
# ==========================================

df.to_csv(
    "../data/final/homestays_prepared.csv",
    index=False
)

print("Prepared dataset saved successfully.")
print(f"Final Shape: {df.shape}")

Prepared dataset saved successfully.
Final Shape: (811, 45)
